# धडा ०५ - एजेंटिक RAG


## सेटअप

हा नोटबुक Microsoft Agent Framework वापरून Agentic RAG (Retrieval-Augmented Generation) नमुना दाखवतो.

**पूर्वशर्त:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तुमचा Azure AI Search सेवा endpoint
- `AZURE_SEARCH_API_KEY` — तुमचा Azure AI Search API key
- पर्यावरणीय चलांच्या माध्यमातून Azure OpenAI deployment कॉन्फिगर केलेले
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजंटिक RAG म्हणजे काय?

पारंपरिक RAG एक निश्चित प्रक्रिया पालन करते: दस्तऐवज मिळवा, नंतर प्रतिसाद तयार करा. **एजंटिक RAG** हे पुढे जाते आणि एजंटला माहिती मिळवण्यासाठी **कधी** आणि **कसे** या बाबतीत स्वायत्तता देते.

एजंटिक RAG सह, एजंट खालील गोष्टी करू शकतो:
- प्रश्नाचे उत्तर देण्यापूर्वी मिळवणे आवश्यक आहे की नाही हे **निर्णय** घेणे
- कोणता डेटा स्रोत किंवा साधन विचारायचे ते **निवडणे**
- मिळवलेल्या निकालांचे **मूल्यमापन** करणे आणि जर पहिल्या प्रयत्नामध्ये अपुरे असेल तर फॉलो-अप मिळवणे करणे
- अनेक मिळवण्याच्या टप्प्यांमधील माहिती एकत्र करून सुसंगत उत्तर तयार करणे

हा एजंटला एक स्थिर retrieve-then-generate प्रक्रियेपेक्षा अधिक लवचिक आणि अचूक बनवतो.


## शोध साधन तयार करणे

Agentic RAG मध्ये, बाह्य डेटा स्रोतांना **साधने** म्हणून वळवले जाते ज्यांना एजंट आवश्यकतेनुसार वापरू शकतो. यामुळे एजंटला पुनर्प्राप्ती फक्त आणखी एक क्रिया म्हणून घेतली जाते, अशी अनिवार्य पायरी नव्हे.

खालिलमध्ये आपण प्रवास ज्ञान भांडार परिभाषित करतो व ते एक साधन म्हणून प्रस्तुत करतो ज्याला एजंट गंतव्य माहिती शोधण्यासाठी कॉल करू शकतो.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजंट तयार करणे

आता आपण एक एजंट तयार करतो ज्याला **उत्तर देण्यापूर्वी नेहमी माहिती मिळवण्याचे निर्देश दिलेले आहेत**. एजंट आपल्या स्वतःच्या प्रशिक्षण डेटावर अवलंबून न राहता ज्ञानाधारावर आधारित आपले प्रतिसाद स्थापन करण्यासाठी `search_travel_knowledge` साधन वापरतो.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्ती शोध — मेकर-चेककर नमुना

Agentic RAG चा एक मुख्य फायदा म्हणजे **पुनरावृत्ती शोध**. एजंट त्याच्या प्रथम शोधलेल्या माहितीची पुष्टी करणे, सुधारणा करणे किंवा विस्तार करण्यासाठी एकापेक्षा अधिक राउंड्समध्ये शोध घेऊ शकतो — हे "मेकर-चेककर" कार्यप्रवाहासारखे आहे:

1. **मेकर पायरी**: एजंट प्राथमिक माहिती शोधून उत्तराचा मसुदा तयार करतो.
2. **चेककर पायरी**: एजंट तपशिलांची पुष्टी करण्यासाठी किंवा रिकाम्या जागा भरण्यासाठी अतिरिक्त शोध करतो.

खाली, एजंटला असे एक प्रश्न विचारले जाते जे अनेक ठिकाणांची तुलना करणे अपेक्षित करते, ज्यामुळे त्याला अनेक वेळा शोध घेण्यास प्रवृत्त केले जाते.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून **Agentic RAG** प्रणाली कशी तयार करायची हे शिकलात:

- **Agentic RAG** एजंट्सना माहिती शोधण्याचा निर्णय स्वायत्तपणे घेण्याची परवानगी देते, ज्यामुळे शोध प्रक्रिया निश्चित न राहता गतिशील होते.
- **टूल्स डेटास्रोत म्हणून**: बाह्य ज्ञानाधारे (जसे Azure AI Search) टूल्स म्हणून वावरण्यात येतात ज्यांना एजंट वापरू शकतो.
- **आवर्ती शोध**: मेकर-चेकअर पॅटर्न एजंटला अनेक शोध फेरींमध्ये काम करण्याची अनुमती देते — शोधणे, पडताळणी करणे आणि शुध्दता करणे — अंतिम उत्तर तयार करण्यापूर्वी.

उत्पादनात, तुम्ही इन-मेमरी `TRAVEL_KNOWLEDGE_BASE` ची जागा वास्तविक Azure AI Search निर्देशांकाने बदलाल, ज्याने मोठ्या प्रमाणावर प्रवास दस्तऐवज शोधणे सुलभ होईल.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
